# Regime-Switching Equity Long/Short Strategy**A systematic strategy that detects market regimes with a Hidden Markov Model and switches between momentum and mean-reversion signals based on the regime.**Built and tested on 503 S&P 500 stocks over 26 years (2000-present).---This notebook documents the **full research journey**, including the approaches that failed and the bugs I caught, because the honest research process is the point. Most of what I tried did not work; the final strategy emerged from working through those failures.**Final result:** statistically significant (signal t=3.1), net Sharpe 0.64-0.75 after transaction costs, 6.4x gross cumulative return.**Journey:** single stock -> small universe -> mean-reversion (caught a look-ahead bug) -> breadth test (collapsed) -> signal decay -> went monthly -> expanded to 2000 -> full 503-stock universe -> HMM regime detection (broke, fixed with SPY) -> found regime-dependence -> switching strategy (significant) -> portfolio + costs (tradeable).

## Setup

In [ ]:
!pip install yfinance hmmlearn

In [ ]:
import pandas as pd, numpy as np, yfinance as yfimport matplotlib.pyplot as pltfrom hmmlearn.hmm import GaussianHMMimport requests

## Stage 1 — First attempt: Momentum on a single stock (FAILED)I started simple: test whether momentum (rising stocks keep rising) works, on a small set of stocks, daily.**Result: weak, and it actually showed short-term REVERSAL, not momentum.** Lesson: daily momentum at this horizon doesn't work here, and I need more breadth and the right horizon.

In [ ]:
tickers_small = ["AAPL","MSFT","GOOG","AMZN","META","NVDA","TSLA","JPM","V","WMT",                 "JNJ","PG","XOM","HD","BAC","KO","PFE","CSCO","INTC","T"]d_small = yf.download(tickers_small, start="2021-01-01", end="2024-01-01")["Close"]returns = d_small.pct_change()momentum = returns.rolling(window=60).mean()          # 60-day momentumfuture_returns = returns.shift(-1)                     # next-day returndaily_ic = momentum.corrwith(future_returns, axis=1)t = daily_ic.mean() / (daily_ic.std() / np.sqrt(daily_ic.count()))print("Momentum IC:", daily_ic.mean(), " t:", t)       # weak / not significant

## Stage 2 — Mean-reversion on a broader universe + a LOOK-AHEAD BIAS bug I caughtI pivoted to mean-reversion (recent losers bounce) on a larger, diversified universe (2018-present).**Important:** my first version produced an *impossible* IC (around -0.78) because I had accidentally overwritten a variable so the signal overlapped the future return, a classic **look-ahead bias** bug. "Too good to be true" = a bug. The fix: keep the signal (past) and the future return cleanly separate, with no overlapping window.

In [ ]:
tickers_mid = [ "AAPL","MSFT","GOOG","GOOGL","AMZN","META","NVDA","AVGO","ORCL","CRM","ADBE","CSCO","ACN","AMD","INTC","IBM","QCOM","TXN","INTU","NOW", "JPM","BAC","WFC","GS","MS","C","AXP","BLK","SCHW","SPGI","CB","PGR","MMC","V","MA", "JNJ","UNH","LLY","ABBV","MRK","PFE","TMO","ABT","DHR","BMY","AMGN","CVS","MDT","ISRG", "WMT","PG","KO","PEP","COST","MCD","NKE","SBUX","TGT","LOW","HD","DIS","CMCSA","NFLX","TMUS","T","VZ", "CAT","BA","GE","HON","UNP","UPS","RTX","LMT","DE","MMM","EMR","ETN", "LIN","SHW","APD","AMT","PLD","EQIX","SPG", "ABNB","UBER","PYPL","COIN","SNOW","PANW","CRWD","DDOG","ADP","BKNG","MDLZ","CL","KMB","GILD","REGN","VRTX", "XOM","CVX","COP","SLB","EOG","PSX","MPC","VLO","OXY","WMB","KMI","HAL","DVN","HES","BKR"]d_mid = yf.download(tickers_mid, start="2018-01-01")["Close"]d_mid = d_mid.dropna(axis=1, how="all")print("universe shape:", d_mid.shape)# CORRECT, no-look-ahead construction: signal from PAST, future return cleanly aheaddaily_returns = d_mid.pct_change()        # 1-day returns  -> for the future returnreversal = -1 * d_mid.pct_change(5)       # 5-day reversal signal (from past data)future_returns = daily_returns.shift(-1)  # next day's return (no overlap with signal)daily_ic = reversal.corrwith(future_returns, axis=1)t = daily_ic.mean() / (daily_ic.std() / np.sqrt(daily_ic.count()))print("Reversal IC:", daily_ic.mean(), " t:", t)

### Horizon sweep (which signal/prediction horizon works?)I swept several signal and prediction horizons rather than guessing one. The prediction horizon must match the signal's timescale.

In [ ]:
for i in [1,3,5,10]:    for j in [1,3,5,10]:        sig = -1 * d_mid.pct_change(i)        fut = d_mid.pct_change(j).shift(-j)        ic = sig.corrwith(fut, axis=1, method="spearman")        t = ic.mean() / (ic.std() / np.sqrt(ic.count()))        print(f"signal={i}d, pred={j}d  |  rankIC={ic.mean():.4f}, t={t:.2f}")

### Signal decay (worked historically, dead recently)Splitting a candidate by period revealed it worked in 2018-2022 but had **decayed** by 2023-present, a key reason to always check recent performance, not just the full sample.

In [ ]:
sig = -1 * d_mid.pct_change(1)fut = d_mid.pct_change(1).shift(-1)daily_ic = sig.corrwith(fut, axis=1, method="spearman")early  = daily_ic.loc["2018":"2022"]recent = daily_ic.loc["2023":]print("Early 2018-2022 : IC", early.mean(),  " t", early.mean()/(early.std()/np.sqrt(early.count())))print("Recent 2023-now : IC", recent.mean(), " t", recent.mean()/(recent.std()/np.sqrt(recent.count())))

## Stage 3 — Full universe (503 S&P 500 stocks) + long history (2000-present)Two lessons drove this: (1) I needed more **breadth** (stocks) and more **crash periods** (years) for statistical power, and (2) a cleaner, standard setup. So I pulled the full S&P 500 from Wikipedia and 26 years of data.I also pull **SPY separately** as a clean market signal for regime detection (explained below).

In [ ]:
# scrape the S&P 500 tickers from Wikipedia (browser header avoids a 403 block)url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})sp500 = pd.read_html(response.text)[0]tickers = [t.replace(".", "-") for t in sp500["Symbol"].tolist()]print(len(tickers), "tickers")# 503 stocks for the SIGNALSdata = yf.download(tickers, start="2000-01-01")["Close"]data = data.dropna(axis=1, how="all")print("data shape:", data.shape)

## Stage 4 — Regime detection with a Hidden Markov Model (HMM)A regime is a *hidden* market state (calm / normal / crash) you can't observe directly, you only see returns and volatility. An HMM infers these hidden states from the data, which is why it beats an arbitrary volatility cutoff.**A bug I caught:** my first HMM used the *average of all 503 stocks* as the market signal, but most didn't exist before 2012, so the early signal was noise and the regime plot was a scattered mess. I spotted it by plotting. The fix: use **SPY** (clean index data back to 2000) as the market signal.

In [ ]:
# CLEAN market signal from SPY (not averaging 503 stocks)spy = yf.download("SPY", start="2000-01-01")["Close"]spy_ret = spy.pct_change()spy_vol = spy_ret.rolling(21).std()           # 21-day volatilityfeatures = pd.DataFrame({"ret": spy_ret.squeeze(),                         "vol": spy_vol.squeeze()}).dropna()print("features shape:", features.shape)

In [ ]:
# fit a 3-state Gaussian HMM on [return, volatility]model = GaussianHMM(n_components=3, covariance_type="full", n_iter=1000, random_state=42)model.fit(features[["ret","vol"]].values)features["regime"] = model.predict(features[["ret","vol"]].values)# characterize the regimes (labels 0/1/2 are ARBITRARY - must check each fit)print(features.groupby("regime")[["ret","vol"]].mean())print(features["regime"].value_counts())# -> crash = negative return + high vol ; calm = positive return + low vol ; normal = in between

### Validate the regimes visuallyThe crash regime should cluster at known crises (2002 dot-com, 2008 crisis, 2020 COVID, 2022 selloff). This plot is the proof the detector works.

In [ ]:
market = spy.squeeze().loc[features.index]reg = features["regime"]# NOTE: set colors by what the characterization above showed (crash/calm/normal)colors = {0: "red", 1: "green", 2: "gray"}      # adjust if your fit labels differlabels = {0: "crash", 1: "calm", 2: "normal"}plt.figure(figsize=(14,6))for r in [0,1,2]:    m = reg == r    plt.scatter(market.index[m], market[m], c=colors[r], s=4, label=labels[r])plt.legend(); plt.title("SPY over time, colored by HMM regime"); plt.xlabel("Date"); plt.ylabel("Market level")plt.show()

## Stage 5 — Monthly signals + how each signal behaves by regimeI work at **monthly** frequency (the standard for cross-sectional equity signals: less noise, realistic costs, correct timescale). Signals: textbook **12-1 momentum** and **1-month reversal**.Then the key test, **conditional analysis**: measure each signal's IC *separately within each regime*.

In [ ]:
monthly = data.resample("ME").last()monthly_returns = monthly.pct_change()mom = monthly.shift(1) / monthly.shift(12) - 1    # 12-1 momentum (skip last month)rev = -1 * monthly.pct_change(1)                   # 1-month reversalfuture = monthly_returns.shift(-1)                 # next month's return# monthly regime + probabilities from the clean HMMmonthly_regime = features["regime"].resample("ME").last()probs = model.predict_proba(features[["ret","vol"]].values)# column order follows the regime numbers from the characterization aboveprobs_df = pd.DataFrame(probs, index=features.index, columns=["p_crash","p_calm","p_normal"])monthly_probs = probs_df.resample("ME").last()print(monthly_regime.value_counts())

In [ ]:
mom_ic = mom.corrwith(future, axis=1, method="spearman")rev_ic = rev.corrwith(future, axis=1, method="spearman")for name, ic in [("Momentum", mom_ic), ("Reversal", rev_ic)]:    print(f"\n{name}:")    for r, label in [(1,"calm"), (2,"normal"), (0,"crash")]:   # map to this fit's labels        months = monthly_regime[monthly_regime == r].index        part = ic[ic.index.isin(months)]        print(f"  {label}: IC={part.mean():.4f}  t={part.mean()/(part.std()/np.sqrt(part.count())):.2f}  n={part.count()}")# Finding: reversal is significant in NORMAL regimes; momentum is positive in CALM regimes.

## Stage 6 — The regime-switching strategyUse each signal only where it has edge: **reversal in normal regimes, momentum in calm regimes, flat in crash** (no proven edge there, and sitting out crashes avoids losses).Momentum and reversal are opposite bets, so naively averaging them cancels out; conditioning on the regime is what makes them work.

In [ ]:
strat = mom.copy(); strat[:] = np.nan              # start flat (no position)calm_months   = monthly_regime[monthly_regime == 1].index   # calm  -> momentumstrat.loc[strat.index.isin(calm_months)]   = mom.loc[mom.index.isin(calm_months)]normal_months = monthly_regime[monthly_regime == 2].index   # normal -> reversalstrat.loc[strat.index.isin(normal_months)] = rev.loc[rev.index.isin(normal_months)]# crash months stay NaN = flatstrat_ic = strat.corrwith(future, axis=1, method="spearman")print("Strategy IC:", strat_ic.mean(),      " t:", strat_ic.mean()/(strat_ic.std()/np.sqrt(strat_ic.count())),      " n:", strat_ic.count())        # significant (t > 2)

## Stage 7 — Portfolio construction (dollar-neutral, signal-weighted)Turn the signal into an actual portfolio:- **Dollar-neutral** (demean the signal each month so longs balance shorts, no market bet, isolates alpha)- **Signal-weighted** (normalize so position size reflects signal strength)Then measure annualized Sharpe and the cumulative return (equity curve).

In [ ]:
weights = strat.sub(strat.mean(axis=1), axis=0)           # demean -> dollar-neutralweights = weights.div(weights.abs().sum(axis=1), axis=0)  # normalize -> signal-weightedport_ret = (weights * future).sum(axis=1)                 # monthly portfolio returnprint("Mean monthly return:", port_ret.mean())print("Monthly t-stat:", port_ret.mean()/(port_ret.std()/np.sqrt(port_ret.count())))annual_sharpe = (port_ret.mean()/port_ret.std()) * np.sqrt(12)print("Annualized Sharpe:", annual_sharpe)cumulative = (1 + port_ret.fillna(0)).cumprod()print("Total cumulative return:", cumulative.iloc[-1] - 1)plt.figure(figsize=(12,5))cumulative.plot()plt.title("Strategy Cumulative Return (dollar-neutral long/short, signal-weighted)")plt.ylabel("Growth of $1"); plt.xlabel("Date"); plt.show()

## Stage 8 — Transaction costs (is it actually tradeable?)Gross returns are fantasy; a strategy is only real if it survives trading costs. Cost = turnover x cost-per-trade. I test at 10 bps (realistic) and 20 bps (conservative).

In [ ]:
turnover = weights.diff().abs().sum(axis=1)print("Average monthly turnover:", turnover.mean())net_ret    = port_ret - turnover * 0.0010    # 10 bpsnet_ret_20 = port_ret - turnover * 0.0020    # 20 bps (conservative)print("GROSS annual Sharpe:", (port_ret.mean()/port_ret.std())*np.sqrt(12))print("NET   annual Sharpe (10bps):", (net_ret.mean()/net_ret.std())*np.sqrt(12))print("NET   annual Sharpe (20bps):", (net_ret_20.mean()/net_ret_20.std())*np.sqrt(12))

## ConclusionA **statistically significant, cost-aware, tradeable** regime-switching long/short equity strategy:- Signal t-stat ~3.1; portfolio t-stat ~4.4- Net Sharpe ~0.64-0.75 after realistic transaction costs- ~6.4x gross cumulative return over 26 years- Grows steadily through 2008 and COVID (dollar-neutral + flat in crash)**Key insight:** signals are regime-dependent (momentum in calm, reversal in normal); naively combining them cancels out, but conditioning on the detected regime makes them work.**Honest caveats:** survivorship bias (today's S&P 500 applied back to 2000); crash periods are not traded (too few to find a reliable signal); results depend on the cost assumption; pre-slippage.**Next steps:** volatility-scaling, a crash-specific defensive signal, turnover reduction, a point-in-time universe.